In [ ]:
import osmnx as ox
from osmnx.features import features_from_bbox
import geopandas as gpd
from src.feature_building_utils import *
from src.geometric_utils import *
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import toml

In [ ]:
df = pd.read_parquet('data/processed_data/S3-approx-coordinates.parquet')

In [ ]:
docs = toml.load("documentation/feature_docs.toml")

In [ ]:
tags_man_made = {"man_made": True}
gdf_man_made = features_from_bbox(BBOX, tags_man_made)

In [ ]:
feature_name = "close2chimney_150"

df[feature_name] = df.apply(
    lambda row: is_close_to(
        features=gdf_man_made,
        point=Point(row.x, row.y),
        threshold=150,
        type_column="man_made",
        types=["chimney"]
    ),
    axis=1
).astype(int)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "'Dummy' variable indicating whether the point is less than 150m away from a chimney"
docs[feature_name]["type"] = "boolean"
docs[feature_name]["range"] = {int(df[feature_name].min()), int(df[feature_name].max())}
docs[feature_name]["created_on"] = "osmnx_man_made.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "close2water_tap_75"

df[feature_name] = df.apply(
    lambda row: is_close_to(
        features=gdf_man_made,
        point=Point(row.x, row.y),
        threshold=75,
        type_column="man_made",
        types=["water_tap"]
    ),
    axis=1
).astype(int)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "'Dummy' variable indicating whether the point is less than 75m away from a water_tap"
docs[feature_name]["type"] = "boolean"
docs[feature_name]["range"] = {int(df[feature_name].min()), int(df[feature_name].max())}
docs[feature_name]["created_on"] = "osmnx_man_made.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
with open("documentation/feature_docs.toml", "w") as f:
    toml.dump(docs, f)

In [ ]:
df.to_parquet('data/processed_data/S3-approx-coordinates.parquet')